In [1]:
!pip install pillow

In [7]:
##Read image in
from PIL import Image
from pathlib import Path

image_path = Path(r"D:\Images\ivotuk\2011\01\ivotuk_2011_01_01_090000.jpg")
img= Image.open(image_path)
print (img)

<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x480 at 0x18B92835A90>


In [10]:
##Turn image into numbers
import numpy as np

arr = np.asarray(img,dtype=np.float32)
print (arr.shape)
print (arr[0,0])

(480, 640, 3)
[1. 1. 1.]


In [15]:
#Convert RBG to Luminence
R = arr[:,:,0]
G = arr[:,:,1]
B=arr[:,:,2]

luminence = 0.2126*R+0.7152*G+0.0722*B
print (luminence)

mean=float(np.mean(luminence))
print(mean)

[[1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
2.063363552093506


In [28]:
##Convert to brightness/stats function
def get_brightness (image_path):
    with Image.open(image_path) as img:
        
        arr = np.asarray(img,dtype=np.float32)

    R = arr[:,:,0]
    G = arr[:,:,1]
    B=arr[:,:,2]

    luminence = 0.2126*R+0.7152*G+0.0722*B

    return{
        "mean":float(np.mean(luminence)),
        "median":float(np.median(luminence)),
        "p10":float(np.percentile(luminence, 10)),
        "p25":float(np.percentile(luminence, 25)),
        "p75":float(np.percentile(luminence, 75)),
        "p90":float(np.percentile(luminence, 90)),
        "frac_above_50":float(np.mean(luminence > 50)),
        "frac_above_75":float(np.mean(luminence > 75)),
        "frac_below_25":float(np.mean(luminence < 25)),
    }

In [17]:
#Create classification rules
def classify_image(stats):
    mean = stats["mean"]
    median = stats ["median"]
    bright_fraction=stats["frac_above_50"]
    dark_fraction=stats["frac_below_25"]

    if median < 30:
        return "REJECT"

    if mean < 40 and bright_fraction <0.20:
        return "REJECT"

    if median < 45:
        return "Review"

    return "Accept"

In [20]:
stats = get_brightness(image_path)

classification =classify_image(stats)

print(classification)

REJECT


In [26]:
#Loop through ONE folder
Img_folder=Path(r"D:\Images\ivotuk\2011\01")

results=[]

for image_path in Img_folder.rglob("*"):
    try:
        stats = get_brightness(image_path)
        classification =classify_image(stats)

        results.append({
        "file": str(image_path),
        "classification": classification,
        **stats,
            })

    except Exception as e:
        print(f"Could not process {image_path}: {e}")

In [23]:
!pip install pandas
import pandas as pd

df=pd.DataFrame(results)
df.to_csv(
    "image_brightness_results.csv",
    index=False,
)

print (df)

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------- ----------------------------- 2.6/9.8 MB 22.0 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.8 MB 36.7 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 24.8 MB/s  0:00:00
                                                   file classification  \
0     D:\Images\ivotuk\2011\01\ivotuk_2011_01_01_090...         REJECT   
1     D:\Images\ivotuk\2011\01\ivotuk_2011_01_01_091...         REJECT   
2     D:\Images\ivotuk\2011\01\ivotuk_2011_01_01_093...         REJECT   
3     D:\Images\ivotuk\2011\01\ivotuk_2011_01_01_094...         REJECT   
4     D:\Images\ivotuk\2011\01\ivotuk_2011_01_01_100...         REJECT   
...                                                 ...            ...   
1142  D:\Images\ivotuk\2011\01\ivotuk_2011_01_31_170...         Accept   
1143  D:\Images\ivotuk\2011\01\ivotuk_2011_01_31_171...         REJECT   
1144  D:\Images\ivotuk\2011\01\ivotuk

In [27]:
#Loop through all images
root= Path(r"D:\Images")

for image_path in root.rglob("*"):
    if not image_path.is_file():
        continue

    if image_path.suffix.lower() not in (".jpg", ".jpeg"):
        continue

    try:
        stats = get_brightness(image_path)
        classification =classify_image(stats)

        results.append({
        "file": str(image_path),
        "classification": classification,
        **stats,
            })

    except Exception as e:
        print(f"Could not process {image_path}: {e}")

KeyboardInterrupt: 

In [29]:
#Loop through all images and sort
import shutil

class_folders={"Accept", "Reject", "Review"}

for image_path in root.rglob("*"):
    if not image_path.is_file():
        continue

    if image_path.suffix.lower() not in (".jpg", ".jpeg"):
        continue

    if any(part in class_folders for part in image_path.parts):
        continue

    try:
        stats = get_brightness(image_path)
        classification =classify_image(stats)

        for folder_name in class_folders:
            (image_path.parent / folder_name).mkdir(exist_ok=True)

        dest_path=image_path.parent/classification/image_path.name
        shutil.copy2(image_path, dest_path)

        results.append({
        "file": str(image_path),
        "classification": classification,
        **stats,
            })

    except Exception as e:
        print(f"Could not process {image_path}: {e}")

Could not process D:\Images\ivotuk\2011\05\ivotuk_2011_05_24_123000.jpg: image file is truncated (55 bytes not processed)
